# 00 · Inventario de conversaciones Mantra
Descubre automáticamente los archivos de `MyDrive/mine_chatbot`, formatos, tamaños, columnas y duplicados. No modifica los datos originales.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/ai_assistant
!git clone -q https://github.com/dinatalediego/ai_assistant.git
%cd /content/ai_assistant
!pip -q install pandas openpyxl pyarrow pyyaml

In [ ]:
from pathlib import Path
from src.drive_inventory import build_inventory
DRIVE_ROOT = Path('/content/drive/MyDrive/mine_chatbot')
assert DRIVE_ROOT.exists(), f'No existe: {DRIVE_ROOT}'
inventory = build_inventory(DRIVE_ROOT)
print('Archivos encontrados:', len(inventory))
display(inventory.head(20))
if inventory.empty:
    print('⚠️ No se encontraron archivos dentro de mine_chatbot. Revisa que los lotes hayan terminado de subir y que Drive esté montado con la misma cuenta.')

In [ ]:
if not inventory.empty:
    resumen = (inventory.groupby('suffix', dropna=False)
               .agg(archivos=('name','size'), bytes=('bytes','sum'), soportados=('supported','sum'))
               .sort_values('archivos', ascending=False))
    display(resumen)
else:
    print('Sin archivos para resumir.')

In [ ]:
if not inventory.empty:
    duplicates = inventory[inventory.duplicated('sha256', keep=False)].sort_values('sha256')
else:
    duplicates = inventory.copy()
print('Archivos que participan en duplicados exactos:', len(duplicates))
if not duplicates.empty:
    display(duplicates[['name','path','bytes','sha256']].head(50))

In [ ]:
from collections import Counter
parsed = inventory[inventory['supported'] & inventory['columns'].notna()] if not inventory.empty else inventory
column_sets = Counter(tuple(x) for x in parsed['columns']) if not parsed.empty else Counter()
if not column_sets:
    print('No se detectaron tablas parseables todavía. Revisa la columna suffix/error del inventario: ahí veremos el formato real exportado por Mantra.')
for cols, n in column_sets.most_common(10):
    print(f'\n{n} archivo(s) · {len(cols)} columnas')
    print(cols)

In [ ]:
OUT = Path('/content/drive/MyDrive/mine_chatbot/_analysis_outputs')
OUT.mkdir(exist_ok=True)
inventory.to_csv(OUT/'file_inventory.csv', index=False)
duplicates.to_csv(OUT/'exact_duplicates.csv', index=False)
print('Resultados guardados en:', OUT)